# 08 — C-MAPSS FD002: Regime-Based Normalization Ablation

FD002 has **six operating conditions** instead of FD001's one (confirmed by clustering
`op_setting_1-3`: 6 distinct regimes, exactly matching the literature). This matters a lot for
preprocessing: a sensor's raw reading is driven by *which regime the engine is in* far more
than by degradation - e.g. `sensor_2`'s mean shifts by ~106 units across regimes vs. a ~0.4
within-regime std. Naively normalizing globally (the FD001 approach) lets that condition-driven
offset swamp the actual degradation signal.

This notebook runs the **same two models as FD001** (Gradient Boosting, LSTM) under **two
preprocessing strategies** to make that effect visible rather than just assumed:
- **naive**: exactly FD001's pipeline (global feature selection + global normalization),
  applied unchanged to FD002.
- **regime**: `src/data/cmapss.py::RegimeNormalizer` - k-means clusters the operating settings
  into regimes (fit on train only), then z-scores each sensor within its own regime.

Same RUL-capping, sliding-window, unit-level-split, and NASA-scoring methodology as
[notebooks/07_cmapss_rul_regression.ipynb](07_cmapss_rul_regression.ipynb) - see
[docs/cmapss_results.md](../docs/cmapss_results.md) for that write-up.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.data.cmapss import (
    add_capped_rul,
    build_test_sequences,
    build_train_sequences,
    fit_regime_normalizer,
    load_subset,
    select_informative_columns,
    SENSOR_COLS,
)
from src.data.cmapss_torch import RulSequenceDataset, compute_sequence_train_stats
from src.models.lstm_rul import LstmRul, predict as lstm_predict, train_lstm
from src.models.rul_baseline import summarize_windows, train_gbr
from src.models.rul_metrics import nasa_score, rmse

CMAPSS_DIR = Path.cwd().parent / "data" / "raw" / "cmapss"
WINDOW_SIZE = 30
torch.manual_seed(42)
np.random.seed(42)


## Load FD002, cap RUL, unit-level train/val split

In [2]:
train_df, test_df, test_true_rul = load_subset(CMAPSS_DIR, subset="FD002")
train_df = add_capped_rul(train_df, cap=125)
print(f"train: {train_df['unit_id'].nunique()} units, {len(train_df)} rows | test: {test_df['unit_id'].nunique()} units")

rng = np.random.default_rng(42)
unit_ids = train_df["unit_id"].unique()
rng.shuffle(unit_ids)
n_val_units = int(0.15 * len(unit_ids))
val_units = set(unit_ids[:n_val_units])
train_units = set(unit_ids[n_val_units:])

train_split_df = train_df[train_df["unit_id"].isin(train_units)]
val_split_df = train_df[train_df["unit_id"].isin(val_units)]
print(f"train units: {len(train_units)} | val units: {len(val_units)}")


train: 260 units, 53759 rows | test: 259 units
train units: 221 | val units: 39


## Confirm the 6-regime structure and condition-conflation

(the exploration that motivated this notebook - see markdown intro above)


In [3]:
op_rounded = train_df[["op_setting_1", "op_setting_2", "op_setting_3"]].round(1)
print(f"{op_rounded.drop_duplicates().shape[0]} unique (rounded) op_setting combos -> collapse to 6 true regimes with measurement noise")

sensor2_by_combo = train_df.groupby([op_rounded["op_setting_1"], op_rounded["op_setting_2"]])["sensor_2"].agg(["mean", "std"])
print("\nsensor_2 mean/std by (rounded) regime:")
print(sensor2_by_combo)


7 unique (rounded) op_setting combos -> collapse to 6 true regimes with measurement noise

sensor_2 mean/std by (rounded) regime:
                                 mean       std
op_setting_1 op_setting_2                      
0.0          0.0           642.678721  0.495905
10.0         0.2           604.904635  0.474585
             0.3           604.922524  0.474298
20.0         0.7           607.564851  0.463106
25.0         0.6           536.865069  0.358128
35.0         0.8           555.799163  0.439489
42.0         0.8           549.704881  0.432850


## Pipeline A: naive (FD001-style global normalization, unchanged)

In [4]:
naive_feature_cols = select_informative_columns(train_split_df)
print(f"naive: {len(naive_feature_cols)} feature columns")

naive_train_seq, naive_train_y, _ = build_train_sequences(train_split_df, naive_feature_cols, window_size=WINDOW_SIZE)
naive_val_seq, naive_val_y, _ = build_train_sequences(val_split_df, naive_feature_cols, window_size=WINDOW_SIZE)
naive_test_seq, _ = build_test_sequences(test_df, naive_feature_cols, window_size=WINDOW_SIZE)
print(f"train windows: {naive_train_seq.shape} | val: {naive_val_seq.shape} | test: {naive_test_seq.shape}")


naive: 24 feature columns


train windows: (39424, 30, 24) | val: (6795, 30, 24) | test: (259, 30, 24)


## Pipeline B: regime-normalized

In [5]:
normalizer = fit_regime_normalizer(train_split_df, sensor_cols=SENSOR_COLS, n_regimes=6)
regime_train_split_df = normalizer.transform(train_split_df)
regime_val_split_df = normalizer.transform(val_split_df)
regime_test_df = normalizer.transform(test_df)

regime_feature_cols = select_informative_columns(regime_train_split_df, candidate_cols=SENSOR_COLS)
print(f"regime: {len(regime_feature_cols)} feature columns (sensors only - op_settings already absorbed into regime assignment)")

regime_train_seq, regime_train_y, _ = build_train_sequences(regime_train_split_df, regime_feature_cols, window_size=WINDOW_SIZE)
regime_val_seq, regime_val_y, _ = build_train_sequences(regime_val_split_df, regime_feature_cols, window_size=WINDOW_SIZE)
regime_test_seq, _ = build_test_sequences(regime_test_df, regime_feature_cols, window_size=WINDOW_SIZE)
print(f"train windows: {regime_train_seq.shape} | val: {regime_val_seq.shape} | test: {regime_test_seq.shape}")


regime: 17 feature columns (sensors only - op_settings already absorbed into regime assignment)


train windows: (39424, 30, 17) | val: (6795, 30, 17) | test: (259, 30, 17)


## Train + evaluate both models under both pipelines

In [6]:
def run_pipeline(name, train_seq, train_y, val_seq, val_y, test_seq, n_features):
    # Gradient Boosting
    X_train_flat = summarize_windows(train_seq)
    X_test_flat = summarize_windows(test_seq)
    gbr = train_gbr(X_train_flat, train_y)
    gbr_pred = gbr.predict(X_test_flat)

    # LSTM
    train_mean, train_std = compute_sequence_train_stats(train_seq)
    train_loader = DataLoader(RulSequenceDataset(train_seq, train_y, train_mean, train_std), batch_size=128, shuffle=True)
    val_loader = DataLoader(RulSequenceDataset(val_seq, val_y, train_mean, train_std), batch_size=128)
    test_loader = DataLoader(RulSequenceDataset(test_seq, np.zeros(len(test_seq)), train_mean, train_std), batch_size=128)

    torch.manual_seed(42)
    lstm_model = LstmRul(n_features=n_features, hidden_size=64, num_layers=2, dropout=0.2)
    train_lstm(lstm_model, train_loader, val_loader, epochs=30, lr=1e-3)
    lstm_pred, _ = lstm_predict(lstm_model, test_loader)

    return {
        f"GBR ({name})": {"rmse": rmse(test_true_rul, gbr_pred), "score": nasa_score(test_true_rul, gbr_pred)},
        f"LSTM ({name})": {"rmse": rmse(test_true_rul, lstm_pred), "score": nasa_score(test_true_rul, lstm_pred)},
    }


results = {}
results.update(run_pipeline("naive", naive_train_seq, naive_train_y, naive_val_seq, naive_val_y, naive_test_seq, len(naive_feature_cols)))
results.update(run_pipeline("regime", regime_train_seq, regime_train_y, regime_val_seq, regime_val_y, regime_test_seq, len(regime_feature_cols)))

summary = pd.DataFrame(results).T
summary.columns = ["rmse_cycles", "nasa_score"]
summary


,rmse_cycles,nasa_score
GBR (naive),31.861188,15301.543129
LSTM (naive),31.650280,36962.582954
GBR (regime),29.640683,14419.732025
LSTM (regime),29.629336,17195.949779


## FD001 vs. FD002 in context

In [7]:
print("For reference, FD001 (single operating condition, no regime normalization needed):")
print("  Gradient Boosting: RMSE 17.04, NASA score 582.8")
print("  LSTM:              RMSE 18.89, NASA score 991.4")
print()
print("FD002 (this notebook):")
print(summary.to_string())


For reference, FD001 (single operating condition, no regime normalization needed):
  Gradient Boosting: RMSE 17.04, NASA score 582.8
  LSTM:              RMSE 18.89, NASA score 991.4

FD002 (this notebook):
               rmse_cycles    nasa_score
GBR (naive)      31.861188  15301.543129
LSTM (naive)     31.650280  36962.582954
GBR (regime)     29.640683  14419.732025
LSTM (regime)    29.629336  17195.949779
